## Router Agent
Here we can use the Python SDK to develop a router agent, then save the agent to a config.yaml and run it from there.

The router agent takes an incoming message, combines it with a prompt and the list of branches, and asks an LLM which branch to take.


In [1]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)


In [2]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)


In [3]:
from nat.control_flow.router_agent.register import RouterAgentWorkflow
from nat.llm.nim_llm import NimLLM
from nat.utils.sdk.nat_workflow import NatWorkflow
from nat_router_agent.register import MockCityAdvisorTool
from nat_router_agent.register import MockFruitAdvisorTool
from nat_router_agent.register import MockLiteratureAdvisorTool

llm = NimLLM(
    model_name="nvdev/meta/llama-3.1-70b-instruct",
    temperature=0.0,
    max_tokens=4096,
    name="nim_llm",
)

# Define the branch functions
fruit_advisor = MockFruitAdvisorTool(
    name="fruit_advisor",
)
city_advisor = MockCityAdvisorTool(
    name="city_advisor",
)
literature_advisor = MockLiteratureAdvisorTool(
    name="literature_advisor",
)

# Create the router agent workflow
agent = RouterAgentWorkflow(
    branch_functions=[fruit_advisor, city_advisor, literature_advisor],
    llm=llm,
    verbose=True,
)

nat_workflow = NatWorkflow(
    entrypoint=agent,
)


/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the workflow to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


functions:
  fruit_advisor:
    _type: mock_fruit_advisor
  city_advisor:
    _type: mock_city_advisor
  literature_advisor:
    _type: mock_literature_advisor

llms:
  nim_llm:
    _type: nim
    model: nvdev/meta/llama-3.1-70b-instruct
    max_tokens: 4096
    temperature: 0.0

workflow:
  _type: router_agent
  llm_name: nim_llm
  verbose: true
  branches:
  - fruit_advisor
  - city_advisor
  - literature_advisor



In [5]:
# Test routing to fruit advisor
await nat_workflow.prompt("What yellow fruit should I eat?")


'banana'

In [6]:
# Test routing to city advisor
await nat_workflow.prompt("What city should I visit in the United States?")


'New York'